In [ ]:
%pip install torch ta mplfinance scikit-learn matplotlib pandas

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import os
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error
import mplfinance as mpf
import pandas as pd
import ta

# --- הגדרת המטבע ---
SYMBOL = 'BTCUSDT'
MODEL_TYPE = 'lstm'

# --- 1. חיבור לגוגל דרייב ---
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("Google Drive mounted successfully!")
    BASE_DIR = '/content/drive/MyDrive/CryptoProject'
except ImportError:
    print("Not running in Google Colab. Using local directory.")
    BASE_DIR = os.path.abspath(os.getcwd())

# --- נתיבים ---
DATA_DIR = os.path.join(BASE_DIR, f'processed_data_{MODEL_TYPE}', SYMBOL)
MODEL_DIR = os.path.join(BASE_DIR, 'models')
CSV_PATH = os.path.join(BASE_DIR, 'data', f'{SYMBOL}_5m_data.csv')
MODEL_SAVE_PATH = os.path.join(MODEL_DIR, f'best_{MODEL_TYPE}_model_{SYMBOL}.pth')
DASHBOARD_FILE_PATH = os.path.join(BASE_DIR, f'{MODEL_TYPE}_dashboard_data_{SYMBOL}.csv')
if not os.path.exists(MODEL_DIR):
    os.makedirs(MODEL_DIR)

# --- Hyperparameters ---
BATCH_SIZE = 128
EPOCHS = 30              # ceiling - EarlyStopping ends sooner
LEARNING_RATE = 0.001
HIDDEN_DIM = 64
NUM_LAYERS = 2
DROPOUT = 0.2
EARLY_STOP_PATIENCE = 5
SEQ_LENGTH = 60          # must match preprocessing
TRAIN_SPLIT = 0.8
VAL_SPLIT = 0.1


class Attention(nn.Module):
    def __init__(self, hidden_dim):
        super(Attention, self).__init__()
        self.attention = nn.Linear(hidden_dim, 1, bias=False)

    def forward(self, lstm_out):
        attn_weights = F.softmax(self.attention(lstm_out), dim=1)
        context = torch.sum(attn_weights * lstm_out, dim=1)
        return context, attn_weights

class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=2, dropout=0.2):
        super(LSTMModel, self).__init__()
        self.hidden_dim = hidden_size
        self.num_layers = num_layers
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True,
                            dropout=dropout if num_layers > 1 else 0.0)
        self.attention = Attention(hidden_size)
        self.fc1 = nn.Linear(hidden_size, hidden_size // 2)
        self.dropout = nn.Dropout(dropout)
        self.fc2 = nn.Linear(hidden_size // 2, 1)

    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_dim).to(x.device)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_dim).to(x.device)
        lstm_out, _ = self.lstm(x, (h0, c0))
        context, _ = self.attention(lstm_out)
        out = F.relu(self.fc1(context))
        out = self.dropout(out)
        out = self.fc2(out)
        # No sigmoid: Bollinger %B is not bounded to [0,1].
        return out


class EarlyStopping:
    def __init__(self, patience=5, delta=1e-6):
        self.patience = patience
        self.delta = delta
        self.best_score = None
        self.early_stop = False
        self.counter = 0
        self.best_loss = np.inf

    def __call__(self, val_loss, model):
        score = -val_loss
        if self.best_score is None:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
        elif score < self.best_score + self.delta:
            self.counter += 1
            print(f'  EarlyStopping: {self.counter}/{self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
            self.counter = 0

    def save_checkpoint(self, val_loss, model):
        torch.save(model.state_dict(), MODEL_SAVE_PATH)
        self.best_loss = val_loss


def load_data():
    print(f"Loading data from {DATA_DIR}...")
    X_train = np.load(os.path.join(DATA_DIR, 'X_train.npy'))
    y_train = np.load(os.path.join(DATA_DIR, 'y_train.npy'))
    X_val = np.load(os.path.join(DATA_DIR, 'X_val.npy'))
    y_val = np.load(os.path.join(DATA_DIR, 'y_val.npy'))
    X_test = np.load(os.path.join(DATA_DIR, 'X_test.npy'))
    y_test = np.load(os.path.join(DATA_DIR, 'y_test.npy'))
    return X_train, y_train, X_val, y_val, X_test, y_test


def train():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")

    X_train, y_train, X_val, y_val, X_test, y_test = load_data()

    # Features are ALREADY StandardScaled in preprocessing - do NOT scale again.
    X_train_tensor = torch.tensor(X_train, dtype=torch.float32).to(device)
    y_train_tensor = torch.tensor(y_train, dtype=torch.float32).to(device)
    X_val_tensor   = torch.tensor(X_val,   dtype=torch.float32).to(device)
    y_val_tensor   = torch.tensor(y_val,   dtype=torch.float32).to(device)
    X_test_tensor  = torch.tensor(X_test,  dtype=torch.float32).to(device)

    train_loader = DataLoader(TensorDataset(X_train_tensor, y_train_tensor), batch_size=BATCH_SIZE, shuffle=True)
    val_loader   = DataLoader(TensorDataset(X_val_tensor, y_val_tensor),     batch_size=BATCH_SIZE, shuffle=False)
    test_loader  = DataLoader(TensorDataset(X_test_tensor),                  batch_size=BATCH_SIZE, shuffle=False)

    N_features = X_train.shape[2]
    model = LSTMModel(input_size=N_features, hidden_size=HIDDEN_DIM, num_layers=NUM_LAYERS, dropout=DROPOUT).to(device)
    criterion = nn.HuberLoss(delta=1.0)
    optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3, min_lr=1e-5)
    early_stopping = EarlyStopping(patience=EARLY_STOP_PATIENCE, delta=1e-6)

    print(f"Starting training for {SYMBOL}...")
    train_losses, val_losses = [], []

    for epoch in range(EPOCHS):
        model.train()
        train_loss = 0.0
        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs.squeeze(), y_batch)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * X_batch.size(0)
        train_loss /= len(train_loader.dataset)
        train_losses.append(train_loss)

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                outputs = model(X_batch)
                val_loss += criterion(outputs.squeeze(), y_batch).item() * X_batch.size(0)
        val_loss /= len(val_loader.dataset)
        val_losses.append(val_loss)

        scheduler.step(val_loss)
        print(f'Epoch {epoch+1}/{EPOCHS}, Train Loss: {train_loss:.6f}, Val Loss: {val_loss:.6f} | LR: {optimizer.param_groups[0]["lr"]:.6f}')

        early_stopping(val_loss, model)
        if early_stopping.early_stop:
            print(f"Early stopping at epoch {epoch+1}")
            break

    # --- Evaluation ---
    print("\nEvaluating on Test Set...")
    model.load_state_dict(torch.load(MODEL_SAVE_PATH, map_location=device))
    model.eval()
    predictions = []
    with torch.no_grad():
        for X_batch, in test_loader:
            predictions.extend(model(X_batch).squeeze().cpu().numpy())
    predictions = np.array(predictions).flatten()

    # ========================================================
    #  DOES IT PREDICT?  (Corr + persistence baseline + copy check)
    # ========================================================
    rmse = np.sqrt(mean_squared_error(y_test, predictions))
    corr = np.corrcoef(predictions, y_test)[0, 1]
    base_rmse = np.sqrt(mean_squared_error(y_test[1:], y_test[:-1]))
    base_corr = np.corrcoef(y_test[:-1], y_test[1:])[0, 1]
    improvement = (base_rmse - rmse) / base_rmse * 100
    corr_gain = corr - base_corr

    print(f"\n--- Results (target: Bollinger %B, 5m ahead) ---")
    print(f"{'':22}{'MODEL':>12}{'PERSISTENCE':>14}")
    print(f"{'RMSE':22}{rmse:>12.6f}{base_rmse:>14.6f}")
    print(f"{'Correlation':22}{corr:>12.3f}{base_corr:>14.3f}")
    print(f"\nRMSE vs persistence baseline: {improvement:+.1f}%")
    print(f"Correlation GAIN over pure copying: {corr_gain:+.3f}")
    if corr < 0.1:
        print(">> Correlation near zero: NOT really predicting.")
    elif corr_gain > 0.05 and improvement > 2:
        print(">> Model BEATS persistence and adds REAL value beyond copying.")
    elif improvement > -2:
        print(">> Roughly matches persistence - mostly copies the last value.")
    else:
        print(">> Worse than persistence.")

    # --- 1) Training loss curve ---
    plt.figure(figsize=(10, 5))
    plt.plot(train_losses, label='Train Loss')
    plt.plot(val_losses, label='Validation Loss')
    plt.title(f'{SYMBOL} LSTM (%B) Training Process')
    plt.xlabel('Epochs'); plt.ylabel('Loss')
    plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

    # --- 2) Actual vs Predicted %B ---
    n_plot = min(576, len(y_test))
    plt.figure(figsize=(14, 7))
    plt.plot(y_test[:n_plot], label='Actual %B', color='blue', alpha=0.7)
    plt.plot(predictions[:n_plot], label='Predicted %B', color='red', linewidth=1.2)
    plt.title(f'{SYMBOL} LSTM Prediction Performance (%B)\nRMSE={rmse:.5f} | Corr={corr:.2f} | vs baseline {improvement:+.1f}%')
    plt.xlabel('Time Steps (5m intervals)'); plt.ylabel('Bollinger %B')
    plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

    # --- 3) Candle visualization (%B -> price) ---
    print("\nGenerating Candle Visualization...")
    df_full = pd.read_csv(CSV_PATH)
    df_full['open_time'] = pd.to_datetime(df_full['open_time'])
    df_full.set_index('open_time', inplace=True)

    n = len(df_full)
    val_end = int(n * (TRAIN_SPLIT + VAL_SPLIT))
    test_start_index = val_end + SEQ_LENGTH
    df_test_candles = df_full.iloc[test_start_index : test_start_index + len(predictions)].copy()

    bb = ta.volatility.BollingerBands(df_full['close'], window=20, window_dev=2)
    upper_band = bb.bollinger_hband().iloc[test_start_index : test_start_index + len(predictions)].values
    lower_band = bb.bollinger_lband().iloc[test_start_index : test_start_index + len(predictions)].values
    predicted_price = lower_band + (predictions * (upper_band - lower_band))
    df_test_candles['Predicted_Close'] = pd.Series(predicted_price).rolling(window=3).mean().bfill().values

    ZOOM_SAMPLES = 576
    df_plot = df_test_candles.head(ZOOM_SAMPLES)
    apdict = mpf.make_addplot(df_plot['Predicted_Close'], type='line', color='red', linestyle='--', width=1.5, panel=0)
    mpf.plot(
        df_plot, type='candle', style='yahoo', addplot=apdict, volume=True,
        title=f'{SYMBOL} Prediction vs Actual\nRMSE (%B): {rmse:.5f}',
        ylabel='Price (USDT)', figsize=(14, 8), tight_layout=True
    )

    # --- Dashboard export ---
    print(f"\nExporting {SYMBOL} LSTM results...")
    df_dashboard = df_test_candles[['open', 'high', 'low', 'close', 'volume']].copy()
    df_dashboard['Predicted_Close'] = df_test_candles['Predicted_Close']
    df_dashboard.to_csv(DASHBOARD_FILE_PATH)
    print(f"✅ {SYMBOL} LSTM data saved to {DASHBOARD_FILE_PATH}!")


if __name__ == "__main__":
    train()
